# 210 · Full Grid Analysis — 9 Models, GOOG Jan 2026

**c10x_v2 grid**: 10 (i,mb) pairs × 3 volumes × 2 directions = 60 tasks per model.  
**9 models**: LobS5, S5-120M, S5-4K, S5-360M, CGAN, CST, Historic, Heuristic, ZeroInsertions.  
**2048 samples** per grid point per direction.  
**Data**: Lustre `/lus/lfs1aip2/projects/s5e/lob_pipeline/LOBS5/evalsequences/aggressive_scenario_v3/`

Analyses:
- A. Beta (square-root law) + bootstrap
- B. Master curves + relaxation ratio
- C. Stability (3-method vote)
- D. No-arbitrage scorecard
- E. Hurst exponent of order flow
- F. Propagator G(l)
- G. Spread dynamics
- H. Parameter sensitivity (mb vs beta)
- I. Null baseline (ZeroInsertions)

In [1]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy import stats, optimize
from scipy.signal import savgol_filter
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [2]:
# -- Publication figure style --
SINGLE_W = 520
FULL_W   = 1080
FIG_H    = 400
TEMPLATE = "plotly_white"
FONT     = dict(family="Times New Roman, serif", size=14)
SAVE_DIR = Path("pics_for_210_full_grid_9m")
SAVE_DIR.mkdir(exist_ok=True)

def save_fig(fig, name, w=FULL_W, h=FIG_H):
    fig.write_image(SAVE_DIR / f"{name}.png", width=w, height=h, scale=3)
    fig.write_image(SAVE_DIR / f"{name}.pdf", width=w, height=h)
    print(f"  Saved {name}")

In [3]:
# ── Configuration ────────────────────────────────────────────────────────
TICK_SIZE = 100
MAX_SAMPLES = 2
N_BOOTSTRAP = 1000
N_COND_MSGS = 500
STOCK = 'GOOG'

SAVE_BASE = Path('/lus/lfs1aip2/projects/s5e/lob_pipeline/LOBS5/evalsequences/aggressive_scenario_v3')

# -- Model metadata (order: baselines first, then neural) --
MODEL_META = OrderedDict([
    ('ZeroInsertions',  dict(color='#66A61E', dash='dot',     marker='diamond-open')),
    ('Historic',        dict(color='#999999', dash='dot',     marker='x')),
    ('Heuristic',       dict(color='#D4A017', dash='dashdot', marker='diamond')),
    ('CST',             dict(color='#E07B39', dash='dash',    marker='triangle-up')),
    ('CGAN',            dict(color='#8B4513', dash='longdash',marker='square')),
    ('LobS5',           dict(color='#1B9E77', dash='solid',   marker='circle')),
    ('S5-120M',         dict(color='#D95F02', dash='solid',   marker='triangle-down')),
    ('S5-4K',           dict(color='#7570B3', dash='solid',   marker='star')),
    ('S5-360M',         dict(color='#E7298A', dash='solid',   marker='hexagon')),
])

# -- All 9 model labels (match directory names on Lustre) --
MODEL_LABELS = list(MODEL_META.keys())

# -- Build paths dict automatically --
# Structure: {SAVE_BASE}/{Label}/context_500_{buy|sell}/GOOG/i*_c*_.../exp_*/data_gen/
PATHS = OrderedDict()
for label in MODEL_LABELS:
    PATHS[label] = dict(
        buy=str(SAVE_BASE / label / f'context_500_buy' / STOCK),
        sell=str(SAVE_BASE / label / f'context_500_sell' / STOCK),
    )

# -- Experiment grid (c10x_v2) --
GRID_PAIRS = [
    (3, 5), (5, 5), (9, 5),
    (2, 10), (3, 10), (4, 10),
    (2, 15), (3, 15),
    (1, 20), (2, 20),
]
VOLUMES = [75, 300, 485]

print(f"Stock: {STOCK}")
print(f"Models: {len(MODEL_LABELS)}")
print(f"Grid: {len(GRID_PAIRS)} pairs × {len(VOLUMES)} vols × 2 dirs = {len(GRID_PAIRS)*len(VOLUMES)*2} tasks/model")
print(f"Base: {SAVE_BASE}")

Stock: GOOG
Models: 9
Grid: 10 pairs × 3 vols × 2 dirs = 60 tasks/model
Base: /lus/lfs1aip2/projects/s5e/lob_pipeline/LOBS5/evalsequences/aggressive_scenario_v3


In [4]:
# ── Data I/O helpers ───────────────────────────────────────────────────

def find_latest_exp(folder_path):
    """Find the latest exp_* subfolder inside a grid-point folder."""
    p = Path(folder_path)
    if not p.exists():
        return p  # fallback: maybe data_gen is directly inside
    exps = sorted(p.glob('exp_*'), key=lambda x: x.stat().st_mtime, reverse=True)
    return exps[0] if exps else p


def discover_folders(buy_root, sell_root):
    """Discover experiment folders from directory structure."""
    folders = []
    buy_root = Path(buy_root)
    sell_root = Path(sell_root)
    if not buy_root.exists():
        print(f"  WARNING: {buy_root} not found")
        return pd.DataFrame()
    for bp in sorted(buy_root.iterdir()):
        if not bp.is_dir(): continue
        name = bp.name
        sp = sell_root / name
        if not sp.exists(): continue
        m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(\d+)%', name)
        if not m: continue
        # Resolve exp_* subfolder
        buy_exp = find_latest_exp(bp)
        sell_exp = find_latest_exp(sp)
        folders.append(dict(
            folder=name, i=int(m.group(1)), c=int(m.group(2)),
            mb=int(m.group(3)), vol=int(m.group(4)), cntxt_pct=int(m.group(5)),
            buy_path=str(buy_exp), sell_path=str(sell_exp)))
    return pd.DataFrame(folders)


def load_folder_data(folder_row, max_samples=MAX_SAMPLES):
    """Load orderbook and message data from a folder pair."""
    buy_gen = Path(folder_row['buy_path']) / 'data_gen'
    sell_gen = Path(folder_row['sell_path']) / 'data_gen'
    buy_books, sell_books, buy_msgs, sell_msgs = [], [], [], []
    for gen_dir, books, msgs in [(buy_gen, buy_books, buy_msgs),
                                 (sell_gen, sell_books, sell_msgs)]:
        if not gen_dir.exists(): continue
        for f in sorted(gen_dir.glob('*_orderbook_*_gen_id_0.csv'))[:max_samples]:
            try: books.append(pd.read_csv(f, header=None).values)
            except: pass
        for f in sorted(gen_dir.glob('*_message_*_gen_id_0.csv'))[:max_samples]:
            try: msgs.append(pd.read_csv(f, header=None).values)
            except: pass
    return dict(buy_books=buy_books, sell_books=sell_books,
                buy_msgs=buy_msgs, sell_msgs=sell_msgs,
                n_buy=len(buy_books), n_sell=len(sell_books))


def load_all(paths_dict, label_prefix=""):
    """Load all data for all models."""
    results = OrderedDict()
    for model_name, paths in paths_dict.items():
        print(f"Loading {label_prefix}{model_name}...")
        grid_df = discover_folders(paths['buy'], paths['sell'])
        if grid_df.empty:
            print(f"  No data for {model_name}")
            continue
        data = {}
        for _, row in grid_df.iterrows():
            data[row['folder']] = load_folder_data(row)
        results[model_name] = dict(grid=grid_df, data=data)
        n_buy = sum(d['n_buy'] for d in data.values())
        n_sell = sum(d['n_sell'] for d in data.values())
        print(f"  {len(grid_df)} folders, {n_buy} buy + {n_sell} sell samples")
    return results

In [5]:
# ── Beta (square-root law) helpers ──────────────────────────────────────

def get_midprice_from_book(book_arr):
    """Extract mid-price: (ask_price + bid_price) / 2."""
    ask = book_arr[:, 0].astype(float)
    bid = book_arr[:, 2].astype(float)
    return (ask + bid) / 2.0


def compute_combined_impact(buy_books, sell_books, tick_size=TICK_SIZE):
    """Compute antisymmetrised impact: (buy_return - sell_return) / 2."""
    buy_returns, sell_returns = [], []
    for b in buy_books:
        mid = get_midprice_from_book(b)
        mid = mid[mid > 0]
        if len(mid) < 2: continue
        buy_returns.append((mid - mid[0]) / tick_size)
    for b in sell_books:
        mid = get_midprice_from_book(b)
        mid = mid[mid > 0]
        if len(mid) < 2: continue
        sell_returns.append((mid - mid[0]) / tick_size)
    return buy_returns, sell_returns


def extract_point_cloud(data, grid_df):
    """Extract (Q, I, sigma) point cloud for beta regression."""
    points = []
    for _, row in grid_df.iterrows():
        folder = row['folder']
        if folder not in data: continue
        fd = data[folder]
        buy_rets, sell_rets = compute_combined_impact(fd['buy_books'], fd['sell_books'])
        n_pairs = min(len(buy_rets), len(sell_rets))
        for j in range(n_pairs):
            br, sr = buy_rets[j], sell_rets[j]
            L = min(len(br), len(sr))
            if L < 2: continue
            insert_end = min(row['i'] * (row['mb'] + 1), L - 1)
            combined = (br[insert_end] - sr[insert_end]) / 2.0
            if abs(combined) < 1e-10 or combined < 0: continue
            points.append(dict(Q=row['i'] * row['vol'], I=combined,
                              vol=row['vol'], i=row['i'], mb=row['mb'], sample_id=j))
    return pd.DataFrame(points)


def compute_global_beta(pc_df, daily_vol=1e6, daily_sigma=1.0):
    """Fit through-origin OLS in log-log space."""
    if pc_df.empty: return dict(beta=np.nan, r2=np.nan, n=0)
    x = np.log(pc_df['Q'].values / daily_vol)
    y = np.log(pc_df['I'].values / daily_sigma)
    beta = np.dot(x, y) / np.dot(x, x)
    y_pred = beta * x
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum(y**2)
    return dict(beta=beta, r2=r2, n=len(pc_df))


def bootstrap_beta(pc_df, n_boot=N_BOOTSTRAP, daily_vol=1e6, daily_sigma=1.0):
    """Bootstrap beta by resampling sample_ids."""
    if pc_df.empty: return np.array([])
    sample_ids = pc_df['sample_id'].unique()
    rng = np.random.default_rng(42)
    betas = []
    for _ in range(n_boot):
        boot_ids = rng.choice(sample_ids, size=len(sample_ids), replace=True)
        boot_df = pd.concat([pc_df[pc_df['sample_id'] == sid] for sid in boot_ids], ignore_index=True)
        betas.append(compute_global_beta(boot_df, daily_vol, daily_sigma)['beta'])
    return np.array(betas)

In [6]:
# ── Master curves, relaxation ─────────────────────────────────────────

def compute_master_curve(data_dict, folder, tick_size=TICK_SIZE, n_vol_u=200):
    """Compute sigma-normalised volume-time master curve."""
    i_val = int(re.search(r'i(\d+)', folder).group(1))
    mb_val = int(re.search(r'mb(\d+)', folder).group(1))
    L = i_val * (mb_val + 1)
    buy_rets, sell_rets = compute_combined_impact(
        data_dict['buy_books'], data_dict['sell_books'], tick_size)
    n_pairs = min(len(buy_rets), len(sell_rets))
    if n_pairs == 0: return None
    max_len = max(
        max((len(r) for r in buy_rets), default=0),
        max((len(r) for r in sell_rets), default=0))
    if max_len == 0: return None
    u_grid = np.linspace(0, max_len / L, n_vol_u)
    combined_curves = []
    for j in range(n_pairs):
        br, sr = buy_rets[j], sell_rets[j]
        min_len = min(len(br), len(sr))
        combined = (br[:min_len] - sr[:min_len]) / 2.0
        u_raw = np.arange(min_len) / L
        combined_curves.append(np.interp(u_grid, u_raw, combined))
    curves = np.array(combined_curves)
    return dict(u=u_grid, mean=np.nanmean(curves, axis=0),
                std=np.nanstd(curves, axis=0), n=n_pairs, folder=folder)


def compute_relaxation_ratio(master_curve, u_peak=1.0, u_final=3.0):
    """Compute I_final / I_peak."""
    if master_curve is None: return np.nan
    u, m = master_curve['u'], master_curve['mean']
    peak_idx = np.argmin(np.abs(u - u_peak))
    final_idx = np.argmin(np.abs(u - u_final))
    if m[peak_idx] < 1e-10: return np.nan
    return m[final_idx] / m[peak_idx]

In [7]:
# ── Stability (3-method vote) ─────────────────────────────────────────
TAIL_FRAC = 0.20
SLOPE_THRESH = 0.05
WINDOW_THRESH = 0.03
CONVERGE_THRESH = 0.95

def stability_for_folder(master_curve, u_peak=1.0):
    """3-method stability vote on post-peak master curve."""
    if master_curve is None:
        return dict(stable=False, votes=0, methods=[False, False, False])
    u, m = master_curve['u'], master_curve['mean']
    peak_idx = np.argmin(np.abs(u - u_peak))
    post = m[peak_idx:]
    if len(post) < 10:
        return dict(stable=False, votes=0, methods=[False, False, False])
    n = len(post)
    tail = post[int(n * (1 - TAIL_FRAC)):]
    # Method 1: trailing slope
    x_tail = np.arange(len(tail))
    slope = np.polyfit(x_tail, tail, 1)[0] if len(tail) > 1 else 1.0
    m1 = abs(slope) / (abs(np.mean(tail)) + 1e-10) < SLOPE_THRESH
    # Method 2: two-window comparison
    mid = n // 2
    w1 = np.mean(post[max(0, mid - n // 8):mid + n // 8])
    w2 = np.mean(tail)
    m2 = abs(w1 - w2) / (abs(w1) + 1e-10) < WINDOW_THRESH
    # Method 3: exponential fit
    try:
        def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
        popt, _ = optimize.curve_fit(exp_decay, np.arange(n), post,
                                      p0=[post[0] - post[-1], 0.1, post[-1]], maxfev=5000)
        m3 = (1 - abs(popt[0] * np.exp(-popt[1] * n)) / (abs(popt[2]) + 1e-10)) > CONVERGE_THRESH
    except:
        m3 = False
    votes = sum([m1, m2, m3])
    return dict(stable=votes >= 2, votes=votes, methods=[m1, m2, m3])

In [8]:
# ── Decay function fitting ────────────────────────────────────────────

def fit_decay(u_grid, mean_curve, u_peak=1.0):
    """Fit power-law and exponential decay to post-peak curve."""
    peak_idx = np.argmin(np.abs(u_grid - u_peak))
    post_u = u_grid[peak_idx:] - u_peak
    post_y = mean_curve[peak_idx:]
    if len(post_y) < 5 or post_y[0] < 1e-10:
        return dict(gamma_power=np.nan, gamma_exp=np.nan)
    y_norm = post_y / post_y[0]
    try:
        def power_law(u, gamma, c): return c * (1 + u)**(-gamma)
        mask = post_u > 0
        popt, _ = optimize.curve_fit(power_law, post_u[mask], y_norm[mask],
                                      p0=[0.5, 1.0], maxfev=5000)
        gamma_power = popt[0]
    except:
        gamma_power = np.nan
    try:
        def exp_decay(u, a, b, c): return a * np.exp(-b * u) + c
        popt, _ = optimize.curve_fit(exp_decay, post_u, y_norm,
                                      p0=[0.3, 0.5, 0.7], maxfev=5000)
        gamma_exp = popt[1]
    except:
        gamma_exp = np.nan
    return dict(gamma_power=gamma_power, gamma_exp=gamma_exp)

In [9]:
# ── Hurst Exponent of Order Flow ───────────────────────────────────

def extract_order_signs(msgs_list):
    """Extract order signs: buy MO = +1, sell MO = -1."""
    all_signs = []
    for msgs in msgs_list:
        mask_mo = msgs[:, 1] == 4
        if mask_mo.sum() < 2: continue
        signs = np.where(msgs[mask_mo, 2] == 0, 1, -1)
        all_signs.append(signs)
    return all_signs


def compute_hurst_dfa(signs, max_lag=200):
    """Detrended Fluctuation Analysis for Hurst exponent."""
    if len(signs) < max_lag * 2: return np.nan
    cumsum = np.cumsum(signs - np.mean(signs))
    scales = np.unique(np.logspace(1, np.log10(max_lag), 20).astype(int))
    scales = scales[scales >= 4]
    flucts = []
    for scale in scales:
        n_seg = len(cumsum) // scale
        if n_seg < 1: continue
        F2 = 0
        for seg in range(n_seg):
            segment = cumsum[seg * scale:(seg + 1) * scale]
            x = np.arange(scale)
            trend = np.polyval(np.polyfit(x, segment, 1), x)
            F2 += np.mean((segment - trend)**2)
        flucts.append(np.sqrt(F2 / n_seg))
    if len(flucts) < 3: return np.nan
    log_s = np.log(scales[:len(flucts)])
    log_f = np.log(flucts)
    H, _ = np.polyfit(log_s, log_f, 1)
    return H


def compute_autocorrelation(signs, max_lag=200):
    """Compute autocorrelation C(l) of order signs."""
    signs = np.array(signs, dtype=float)
    signs = signs - signs.mean()
    n = len(signs)
    var = np.var(signs)
    if var < 1e-10: return np.zeros(max_lag)
    acf = np.zeros(max_lag)
    for lag in range(min(max_lag, n - 1)):
        acf[lag] = np.mean(signs[:n - lag] * signs[lag:]) / var
    return acf

In [10]:
# ── Propagator Function G(l) ──────────────────────────────────────

def compute_propagator(msgs_list, books_list, tick_size=TICK_SIZE, max_lag=200):
    """Compute G(l) = E[dp(t+l) * eps(t)]."""
    G_sum = np.zeros(max_lag)
    G_count = np.zeros(max_lag)
    for msgs, books in zip(msgs_list, books_list):
        if len(msgs) < max_lag + 10: continue
        mid = get_midprice_from_book(books) / tick_size
        dp = np.diff(mid)
        eps = np.zeros(len(msgs))
        eps[(msgs[:, 1] == 4) & (msgs[:, 2] == 0)] = 1
        eps[(msgs[:, 1] == 4) & (msgs[:, 2] == 1)] = -1
        n = min(len(dp), len(eps) - 1)
        for lag in range(min(max_lag, n)):
            valid = n - lag
            G_sum[lag] += np.sum(dp[lag:lag + valid] * eps[:valid])
            G_count[lag] += valid
    return np.where(G_count > 0, G_sum / G_count, 0), np.arange(max_lag)

In [11]:
# ── Spread Dynamics During Impact ────────────────────────────────

def compute_spread_trajectory(buy_books, sell_books, n_injection_msgs):
    """Extract bid-ask spread trajectory during impact."""
    all_spreads = []
    for books in buy_books + sell_books:
        if len(books) < 10: continue
        spread = (books[:, 0].astype(float) - books[:, 2].astype(float)) / TICK_SIZE
        spread[spread <= 0] = np.nan
        spread[spread > 100] = np.nan
        all_spreads.append(spread)
    if not all_spreads: return None
    max_len = max(len(s) for s in all_spreads)
    L = max(n_injection_msgs, 1)
    u_grid = np.linspace(0, max_len / L, 200)
    interp_spreads = []
    for s in all_spreads:
        u_raw = np.arange(len(s)) / L
        valid = ~np.isnan(s)
        if valid.sum() < 5: continue
        interp_spreads.append(np.interp(u_grid, u_raw[valid], s[valid]))
    if not interp_spreads: return None
    arr = np.array(interp_spreads)
    return dict(u=u_grid, mean=np.nanmean(arr, axis=0),
                std=np.nanstd(arr, axis=0), n=len(interp_spreads))

---
## 1. Load Data

In [12]:
print('=' * 60)
print(f'Loading {STOCK} Jan 2026 — 9 models, c10x_v2 grid')
print('=' * 60)
R = load_all(PATHS, label_prefix=f'{STOCK}/')
print(f'\nLoaded {len(R)} / {len(MODEL_LABELS)} models')

Loading GOOG Jan 2026 — 9 models, c10x_v2 grid
Loading GOOG/ZeroInsertions...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/Historic...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/Heuristic...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/CST...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/CGAN...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/LobS5...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/S5-120M...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/S5-4K...
  30 folders, 60 buy + 60 sell samples
Loading GOOG/S5-360M...
  30 folders, 60 buy + 60 sell samples

Loaded 9 / 9 models


---
## 2. Null Baseline (ZeroInsertions)

In [13]:
from pathlib import Path
SINGLE_W = 520; FULL_W = 1080; FIG_H = 400
SAVE_DIR = Path("pics_for_210_full_grid_9m"); SAVE_DIR.mkdir(exist_ok=True)

def save_fig(fig, name, w=FULL_W, h=FIG_H):
  fig.write_html(SAVE_DIR / f"{name}.html", include_plotlyjs='cdn')
  try:
      fig.write_image(SAVE_DIR / f"{name}.png", width=w, height=h, scale=3)
      fig.write_image(SAVE_DIR / f"{name}.pdf", width=w, height=h)
      print(f"  Saved {name} (.html + .png + .pdf)")
  except Exception:
      print(f"  Saved {name} (.html only — no Chrome on aarch64)")

In [14]:
# ZeroInsertions = historic replay with 0 aggressive orders → drift should be ~0
if 'ZeroInsertions' in R:
    zr = R['ZeroInsertions']
    drifts_buy, drifts_sell = [], []
    for folder, fd in zr['data'].items():
        for b in fd['buy_books']:
            mid = get_midprice_from_book(b); mid = mid[mid > 0]
            if len(mid) >= 2: drifts_buy.append((mid[-1] - mid[0]) / TICK_SIZE)
        for b in fd['sell_books']:
            mid = get_midprice_from_book(b); mid = mid[mid > 0]
            if len(mid) >= 2: drifts_sell.append((mid[-1] - mid[0]) / TICK_SIZE)
    drifts_all = np.array(drifts_buy + drifts_sell)
    print(f'ZeroInsertions: drift = {np.mean(drifts_all):.3f} +/- {np.std(drifts_all):.3f} ticks')
    print(f'  |drift| = {np.mean(np.abs(drifts_all)):.3f}, median |drift| = {np.median(np.abs(drifts_all)):.3f}')
    print(f'  n = {len(drifts_all)}')

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=drifts_all, nbinsx=100, name='Drift distribution',
                               marker_color='#66A61E'))
    fig.add_vline(x=0, line_dash='dash', line_color='black')
    fig.update_layout(title='Null Baseline: Mid-Price Drift (ZeroInsertions)',
                      xaxis_title='Drift (ticks)', yaxis_title='Count',
                      template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
    fig.show()
    save_fig(fig, 'null_baseline_drift', w=SINGLE_W)
else:
    print('ZeroInsertions not found')

ZeroInsertions: drift = -5.275 +/- 1.496 ticks
  |drift| = 5.275, median |drift| = 6.000
  n = 120


Wait expired, Browser is being closed by watchdog.


  Saved null_baseline_drift (.html only — no Chrome on aarch64)


---
## 3. Table 1: Global Beta (square-root law)

In [15]:
# Exclude ZeroInsertions from beta analysis (no aggressive orders)
R_impact = OrderedDict((k, v) for k, v in R.items() if k != 'ZeroInsertions')

rows = []
for label, rd in R_impact.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    res = compute_global_beta(pc)
    boots = bootstrap_beta(pc)
    ci = (np.percentile(boots, 2.5), np.percentile(boots, 97.5)) if len(boots) > 0 else (np.nan, np.nan)
    rows.append(dict(Model=label, beta=res['beta'], R2=res['r2'], N=res['n'],
                     CI_lo=ci[0], CI_hi=ci[1]))
beta_df = pd.DataFrame(rows).sort_values('beta')

print('\n-- Table 1: Global Beta (through-origin OLS) --')
print(f'{"Model":<20} {"beta":>8} {"R2":>8} {"N":>10} {"95% CI":>20}')
for _, r in beta_df.iterrows():
    ci = f"[{r['CI_lo']:.3f}, {r['CI_hi']:.3f}]" if not np.isnan(r['CI_lo']) else '---'
    print(f"{r['Model']:<20} {r['beta']:>8.3f} {r['R2']:>8.3f} {r['N']:>10,.0f} {ci:>20}")


-- Table 1: Global Beta (through-origin OLS) --
Model                    beta       R2          N               95% CI
Heuristic              -0.082    0.169         58     [-0.096, -0.068]
CST                    -0.077    0.268         50     [-0.079, -0.075]
Historic               -0.075    0.148         58     [-0.082, -0.068]
S5-120M                -0.066    0.133         56     [-0.074, -0.059]
LobS5                  -0.065    0.254         58     [-0.103, -0.024]
S5-360M                -0.055    0.104         59     [-0.069, -0.042]
S5-4K                  -0.042    0.055         60     [-0.052, -0.031]
CGAN                    0.135    0.192         58       [0.083, 0.187]


In [16]:
# ── Beta Regression Lines ─────────────────────────────────────────
fig = go.Figure()
for label, rd in R_impact.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    if pc.empty: continue
    res = compute_global_beta(pc)
    meta = MODEL_META.get(label, {})
    x = np.log(pc['Q'].values / 1e6)
    y = np.log(pc['I'].values / 1.0)
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name=label,
        marker=dict(size=3, color=meta.get('color', '#888'), opacity=0.15), showlegend=False))
    x_line = np.array([x.min(), x.max()])
    fig.add_trace(go.Scatter(x=x_line, y=res['beta'] * x_line, mode='lines',
        name=f"{label} (beta={res['beta']:.3f})",
        line=dict(color=meta.get('color', '#888'), dash=meta.get('dash', 'solid'), width=2)))
x_th = np.array([-15, -5])
fig.add_trace(go.Scatter(x=x_th, y=0.5 * x_th, mode='lines', name='Theory beta=0.5',
    line=dict(color='black', dash='dash', width=1)))
fig.update_layout(title=f'Beta Regression (log-log) — {STOCK}, 8 models',
    template=TEMPLATE, font=FONT,
    xaxis_title='ln(Q/V)', yaxis_title='ln(I/sigma)', width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, 'beta_regression_8m')

Wait expired, Browser is being closed by watchdog.


  Saved beta_regression_8m (.html only — no Chrome on aarch64)


In [17]:
# ── Bootstrap Beta Distributions ──────────────────────────────────
fig = go.Figure()
for label, rd in R_impact.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    boots = bootstrap_beta(pc)
    if len(boots) == 0: continue
    meta = MODEL_META.get(label, {})
    fig.add_trace(go.Violin(y=boots, name=label, box_visible=True,
        line_color=meta.get('color', '#888'), meanline_visible=True))
fig.add_hline(y=0.5, line_dash='dash', line_color='black', annotation_text='beta=0.5')
fig.update_layout(title=f'Bootstrap Beta (1000 resamples) — {STOCK}',
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H, yaxis_title='beta')
fig.show()
save_fig(fig, 'bootstrap_beta_8m')

Wait expired, Browser is being closed by watchdog.


  Saved bootstrap_beta_8m (.html only — no Chrome on aarch64)


---
## 4. Master Curves

In [18]:
# ── Master Curves per model (panel grid) ─────────────────────────
n_models = len(R_impact)
n_cols = min(4, n_models)
n_rows = math.ceil(n_models / n_cols)
fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=list(R_impact.keys()),
                    shared_yaxes=True, horizontal_spacing=0.04)
for idx, (label, rd) in enumerate(R_impact.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    meta = MODEL_META.get(label, {})
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        if mc is None: continue
        fig.add_trace(go.Scatter(x=mc['u'], y=mc['mean'], mode='lines',
            line=dict(color=meta.get('color', '#888'), width=1),
            opacity=0.5, showlegend=False), row=row, col=col)
    fig.add_vline(x=1.0, line_dash='dash', line_color='gray', row=row, col=col)
fig.update_layout(title=f'Master Curves per Model — {STOCK}',
    template=TEMPLATE, font=FONT,
    width=FULL_W, height=FIG_H * n_rows // 2)
fig.show()
save_fig(fig, 'master_curves_8m', h=FIG_H * n_rows // 2)

Wait expired, Browser is being closed by watchdog.


  Saved master_curves_8m (.html only — no Chrome on aarch64)


In [19]:
# ── Average Master Curve (all models overlaid) ────────────────────
fig = go.Figure()
for label, rd in R_impact.items():
    meta = MODEL_META.get(label, {})
    all_curves = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        if mc is not None: all_curves.append(mc)
    if not all_curves: continue
    u = all_curves[0]['u']
    means = np.array([c['mean'] for c in all_curves])
    avg = np.nanmean(means, axis=0)
    std = np.nanstd(means, axis=0)
    fig.add_trace(go.Scatter(x=u, y=avg, mode='lines', name=label,
        line=dict(color=meta.get('color', '#888'), dash=meta.get('dash', 'solid'), width=2)))
    fig.add_trace(go.Scatter(
        x=np.concatenate([u, u[::-1]]),
        y=np.concatenate([avg + std, (avg - std)[::-1]]),
        fill='toself', fillcolor=meta.get('color', '#888'), opacity=0.1,
        line=dict(width=0), showlegend=False))
fig.add_vline(x=1.0, line_dash='dash', line_color='gray', annotation_text='u=1')
fig.update_layout(title=f'Average Master Curve — {STOCK}, 8 models',
    template=TEMPLATE, font=FONT,
    xaxis_title='Volume time u = n/L', yaxis_title='I/sigma',
    width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, 'avg_master_curve_8m')

Wait expired, Browser is being closed by watchdog.


  Saved avg_master_curve_8m (.html only — no Chrome on aarch64)


---
## 5. Relaxation Ratio + Stability

In [20]:
# ── Relaxation Ratio ──────────────────────────────────────────────
print('\n-- Relaxation Ratio: I_final / I_peak --')
print(f'{"Model":<20} {"Median r":>10} {"Mean +/- std":>20} {"CV":>8} {"|d| from 2/3":>14}')
relax_data = OrderedDict()
for label, rd in R_impact.items():
    ratios = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
    relax_data[label] = ratios
    if ratios:
        arr = np.array(ratios)
        print(f"{label:<20} {np.median(arr):>10.2f} {np.mean(arr):>8.2f} +/- {np.std(arr):.2f}     "
              f"{np.std(arr) / (abs(np.mean(arr)) + 1e-10):>8.2f} {abs(np.mean(arr) - 2/3):>14.2f}")


-- Relaxation Ratio: I_final / I_peak --
Model                  Median r         Mean +/- std       CV   |d| from 2/3
Historic                   0.15     0.26 +/- 0.28         1.08           0.41
Heuristic                  1.11     1.07 +/- 0.31         0.29           0.40
CST                        1.01     1.69 +/- 2.05         1.21           1.03
CGAN                       3.01 51952.56 +/- 247993.49         4.77       51951.89
LobS5                      1.00     1.40 +/- 1.48         1.06           0.73
S5-120M                    1.28     1.71 +/- 1.75         1.03           1.04
S5-4K                      1.00     0.98 +/- 0.74         0.76           0.31
S5-360M                    1.04     1.11 +/- 1.96         1.77           0.44


In [21]:
# ── Relaxation Ratio Figure ───────────────────────────────────────
fig = go.Figure()
for label, ratios in relax_data.items():
    if not ratios: continue
    meta = MODEL_META.get(label, {})
    fig.add_trace(go.Box(y=ratios, name=label, marker_color=meta.get('color', '#888')))
fig.add_hline(y=2/3, line_dash='dash', line_color='black', annotation_text='2/3 (theory)')
fig.update_layout(title=f'Relaxation Ratio (I_final/I_peak at u=3) — {STOCK}',
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H, yaxis_title='r')
fig.show()
save_fig(fig, 'relaxation_ratio_8m')

Wait expired, Browser is being closed by watchdog.


  Saved relaxation_ratio_8m (.html only — no Chrome on aarch64)


In [22]:
# ── Stability Analysis ────────────────────────────────────────────
print('\n-- Stability (3-method vote) --')
print(f'{"Model":<20} {"Stable":>8} {"Total":>8} {"Fraction":>10}')
for label, rd in R_impact.items():
    stable_count = total_count = 0
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        s = stability_for_folder(mc)
        total_count += 1
        if s['stable']: stable_count += 1
    print(f"{label:<20} {stable_count:>8} {total_count:>8} {stable_count / max(total_count, 1):>10.1%}")


-- Stability (3-method vote) --
Model                  Stable    Total   Fraction
Historic                   22       30      73.3%
Heuristic                  24       30      80.0%
CST                        16       30      53.3%
CGAN                        6       30      20.0%
LobS5                      17       30      56.7%
S5-120M                    19       30      63.3%
S5-4K                      17       30      56.7%
S5-360M                    17       30      56.7%


---
## 6. No-Arbitrage Scorecard

In [23]:
print('\n-- No-Arbitrage Scorecard (5 tests) --')
print(f'{"Model":<15} {"beta_perm":>8} {"r":>8} {"A":>4} {"B":>4} {"C":>4} {"D":>4} {"E":>4} {"Score":>8}')
scorecard_rows = []
for label, rd in R_impact.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    beta = compute_global_beta(pc)['beta']
    ratios, decay_gammas = [], []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
        if mc is not None:
            fd = fit_decay(mc['u'], mc['mean'])
            if not np.isnan(fd['gamma_power']): decay_gammas.append(fd['gamma_power'])
    r_mean = np.mean(ratios) if ratios else np.nan
    beta_perm = beta * (r_mean if not np.isnan(r_mean) else 1.0)
    gamma = np.mean(decay_gammas) if decay_gammas else np.nan
    A = beta < 1
    B = 0.7 <= beta_perm <= 1.3 if not np.isnan(beta_perm) else False
    C = 0.3 <= gamma <= 1.0 if not np.isnan(gamma) else False
    D = 0.5 <= r_mean <= 1.0 if not np.isnan(r_mean) else False
    E = beta <= 1 / (1 + 2 * gamma) if not np.isnan(gamma) else False
    score = sum([A, B, C, D, E])
    scorecard_rows.append(dict(Model=label, beta=beta, beta_perm=beta_perm,
                               r=r_mean, gamma=gamma, score=score))
    chk = lambda v: 'Y' if v else 'N'
    print(f"{label:<15} {beta_perm:>8.2f} {r_mean:>8.2f} "
          f"{chk(A):>4} {chk(B):>4} {chk(C):>4} {chk(D):>4} {chk(E):>4} {score:>6}/5")
scorecard_df = pd.DataFrame(scorecard_rows)


-- No-Arbitrage Scorecard (5 tests) --
Model           beta_perm        r    A    B    C    D    E    Score
Historic           -0.02     0.26    Y    N    N    N    Y      2/5
Heuristic          -0.09     1.07    Y    N    Y    N    Y      3/5
CST                -0.13     1.69    Y    N    N    N    Y      2/5
CGAN             7002.04 51952.56    Y    N    N    N    N      1/5
LobS5              -0.09     1.40    Y    N    N    N    Y      2/5
S5-120M            -0.11     1.71    Y    N    N    N    Y      2/5
S5-4K              -0.04     0.98    Y    N    Y    Y    Y      4/5
S5-360M            -0.06     1.11    Y    N    N    N    Y      2/5


---
## 7. Hurst Exponent of Order Flow

In [24]:
print('\n' + '=' * 60)
print('HURST EXPONENT OF ORDER FLOW')
print('=' * 60)
hurst_results = OrderedDict()
for label, rd in R_impact.items():
    all_signs = []
    for folder, fd in rd['data'].items():
        for msgs in fd.get('buy_msgs', []) + fd.get('sell_msgs', []):
            all_signs.extend(extract_order_signs([msgs]))
    if not all_signs: continue
    all_concat = np.concatenate(all_signs)
    if len(all_concat) < 100: continue
    H_dfa = compute_hurst_dfa(all_concat)
    hurst_results[label] = dict(H_dfa=H_dfa, n_signs=len(all_concat))
    print(f"  {label}: H_DFA={H_dfa:.3f} (n={len(all_concat):,})")
print(f"\n  Empirical benchmark: H ~ 0.7 (Lillo & Farmer 2004)")


HURST EXPONENT OF ORDER FLOW
  Historic: H_DFA=nan (n=594)
  Heuristic: H_DFA=nan (n=594)
  CST: H_DFA=nan (n=664)
  CGAN: H_DFA=nan (n=1,512)
  LobS5: H_DFA=nan (n=508)
  S5-120M: H_DFA=nan (n=608)
  S5-4K: H_DFA=0.527 (n=587)
  S5-360M: H_DFA=nan (n=604)

  Empirical benchmark: H ~ 0.7 (Lillo & Farmer 2004)


In [25]:
if hurst_results:
    fig = go.Figure()
    models = list(hurst_results.keys())
    fig.add_trace(go.Bar(x=models,
        y=[hurst_results[m]['H_dfa'] for m in models],
        marker_color=[MODEL_META.get(m, {}).get('color', '#888') for m in models],
        name='H (DFA)'))
    fig.add_hline(y=0.7, line_dash='dash', line_color='black', annotation_text='Empirical H~0.7')
    fig.add_hline(y=0.5, line_dash='dot', line_color='gray', annotation_text='Random walk H=0.5')
    fig.update_layout(title=f'Hurst Exponent of Generated Order Flow — {STOCK}',
        yaxis_title='H (DFA)', template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
    fig.show()
    save_fig(fig, 'hurst_exponent_8m', w=SINGLE_W)

Wait expired, Browser is being closed by watchdog.


  Saved hurst_exponent_8m (.html only — no Chrome on aarch64)


---
## 8. Propagator G(l)

In [26]:
print('\n' + '=' * 60)
print('PROPAGATOR FUNCTION G(l)')
print('=' * 60)
propagator_results = OrderedDict()
for label, rd in R_impact.items():
    msgs_all, books_all = [], []
    for folder, fd in rd['data'].items():
        for m, b in zip(fd.get('buy_msgs', []), fd.get('buy_books', [])):
            msgs_all.append(m); books_all.append(b)
        for m, b in zip(fd.get('sell_msgs', []), fd.get('sell_books', [])):
            msgs_all.append(m); books_all.append(b)
    if not msgs_all: continue
    G, lags = compute_propagator(msgs_all, books_all, max_lag=200)
    propagator_results[label] = dict(G=G, lags=lags)
    g1 = G[1] if len(G) > 1 else 0
    g10 = G[10] if len(G) > 10 else 0
    g100 = G[100] if len(G) > 100 else 0
    print(f"  {label}: G(1)={g1:.4f}, G(10)={g10:.4f}, G(100)={g100:.4f}")


PROPAGATOR FUNCTION G(l)
  Historic: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  Heuristic: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  CST: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  CGAN: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  LobS5: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  S5-120M: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000
  S5-4K: G(1)=-0.0000, G(10)=0.0000, G(100)=0.0000
  S5-360M: G(1)=0.0000, G(10)=0.0000, G(100)=0.0000


In [27]:
if propagator_results:
    fig = go.Figure()
    for label, pr in propagator_results.items():
        meta = MODEL_META.get(label, {})
        G, lags = pr['G'], pr['lags']
        mask = (lags > 0) & (G > 0)
        if mask.sum() < 3: continue
        fig.add_trace(go.Scatter(x=np.log10(lags[mask]), y=np.log10(G[mask]),
            mode='lines', name=label,
            line=dict(color=meta.get('color', '#888'), dash=meta.get('dash', 'solid'), width=2)))
    l_th = np.logspace(0, 2.3, 50)
    fig.add_trace(go.Scatter(x=np.log10(l_th), y=np.log10(l_th**(-0.5) * 0.1),
        mode='lines', name='l^(-0.5) (theory)',
        line=dict(color='black', dash='dash', width=1)))
    fig.update_layout(title=f'Propagator G(l) — {STOCK}',
        xaxis_title='log10(l)', yaxis_title='log10(G)',
        template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H)
    fig.show()
    save_fig(fig, 'propagator_Gl_8m')

Wait expired, Browser is being closed by watchdog.


  Saved propagator_Gl_8m (.html only — no Chrome on aarch64)


---
## 9. Spread Dynamics During Impact

In [28]:
fig = go.Figure()
for label, rd in R_impact.items():
    meta = MODEL_META.get(label, {})
    all_spread_curves = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        fd = rd['data'][folder]
        n_inj = frow['i'] * (frow['mb'] + 1)
        spread = compute_spread_trajectory(fd['buy_books'], fd['sell_books'], n_inj)
        if spread is not None: all_spread_curves.append(spread)
    if not all_spread_curves: continue
    u = all_spread_curves[0]['u']
    means = np.array([c['mean'] for c in all_spread_curves])
    avg = np.nanmean(means, axis=0)
    fig.add_trace(go.Scatter(x=u, y=avg, mode='lines', name=label,
        line=dict(color=meta.get('color', '#888'), dash=meta.get('dash', 'solid'), width=2)))
fig.add_vline(x=1.0, line_dash='dash', line_color='gray', annotation_text='u=1')
fig.update_layout(title=f'Spread Dynamics During Impact — {STOCK}',
    xaxis_title='Volume time u', yaxis_title='Spread (ticks)',
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, 'spread_dynamics_8m')

Wait expired, Browser is being closed by watchdog.


  Saved spread_dynamics_8m (.html only — no Chrome on aarch64)


---
## 10. Parameter Sensitivity (mb vs beta)

In [29]:
fig = go.Figure()
for label, rd in R_impact.items():
    meta = MODEL_META.get(label, {})
    mb_betas = {}
    for _, frow in rd['grid'].iterrows():
        folder, mb = frow['folder'], frow['mb']
        if folder not in rd['data']: continue
        pc = extract_point_cloud({folder: rd['data'][folder]}, pd.DataFrame([frow]))
        if pc.empty: continue
        res = compute_global_beta(pc)
        if not np.isnan(res['beta']): mb_betas.setdefault(mb, []).append(res['beta'])
    if mb_betas:
        mbs = sorted(mb_betas.keys())
        fig.add_trace(go.Scatter(x=mbs,
            y=[np.mean(mb_betas[m]) for m in mbs],
            mode='lines+markers', name=label,
            line=dict(color=meta.get('color', '#888'), width=2),
            marker=dict(symbol=meta.get('marker', 'circle'), size=8)))
fig.add_hline(y=0.5, line_dash='dash', line_color='black')
fig.update_layout(title=f'beta vs mb — {STOCK}',
    xaxis_title='mb', yaxis_title='beta',
    template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
fig.show()
save_fig(fig, 'beta_vs_mb_8m', w=SINGLE_W)

Wait expired, Browser is being closed by watchdog.


  Saved beta_vs_mb_8m (.html only — no Chrome on aarch64)


In [30]:
# ── beta vs volume ────────────────────────────────────────────────
fig = go.Figure()
for label, rd in R_impact.items():
    meta = MODEL_META.get(label, {})
    vol_betas = {}
    for _, frow in rd['grid'].iterrows():
        folder, vol = frow['folder'], frow['vol']
        if folder not in rd['data']: continue
        pc = extract_point_cloud({folder: rd['data'][folder]}, pd.DataFrame([frow]))
        if pc.empty: continue
        res = compute_global_beta(pc)
        if not np.isnan(res['beta']): vol_betas.setdefault(vol, []).append(res['beta'])
    if vol_betas:
        vols = sorted(vol_betas.keys())
        fig.add_trace(go.Scatter(x=vols,
            y=[np.mean(vol_betas[v]) for v in vols],
            mode='lines+markers', name=label,
            line=dict(color=meta.get('color', '#888'), width=2),
            marker=dict(symbol=meta.get('marker', 'circle'), size=8)))
fig.add_hline(y=0.5, line_dash='dash', line_color='black')
fig.update_layout(title=f'beta vs order volume — {STOCK}',
    xaxis_title='Order volume', yaxis_title='beta',
    template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
fig.show()
save_fig(fig, 'beta_vs_volume_8m', w=SINGLE_W)

Wait expired, Browser is being closed by watchdog.


  Saved beta_vs_volume_8m (.html only — no Chrome on aarch64)


---
## 11. Summary Table + Export

In [31]:
summary_rows = []
for label, rd in R_impact.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    beta_res = compute_global_beta(pc)
    boots = bootstrap_beta(pc)
    ci = (np.percentile(boots, 2.5), np.percentile(boots, 97.5)) if len(boots) > 0 else (np.nan, np.nan)
    ratios, stable_count, total_count = [], 0, 0
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], folder)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
        s = stability_for_folder(mc)
        total_count += 1
        if s['stable']: stable_count += 1
    H = hurst_results.get(label, {}).get('H_dfa', np.nan)
    sc = next((s for s in scorecard_rows if s['Model'] == label), {})
    summary_rows.append(dict(
        Model=label, beta=beta_res['beta'], CI_lo=ci[0], CI_hi=ci[1],
        R2=beta_res['r2'], N=beta_res['n'],
        relaxation_mean=np.mean(ratios) if ratios else np.nan,
        relaxation_std=np.std(ratios) if ratios else np.nan,
        stability_frac=stable_count / max(total_count, 1),
        Hurst_DFA=H,
        no_arb_score=sc.get('score', np.nan),
        n_configs=total_count))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SAVE_DIR / 'summary_statistics.csv', index=False)
print(f'\nSaved summary to {SAVE_DIR / "summary_statistics.csv"}')
print(summary_df.to_string(index=False))


Saved summary to pics_for_210_full_grid_9m/summary_statistics.csv
    Model      beta     CI_lo     CI_hi       R2  N  relaxation_mean  relaxation_std  stability_frac  Hurst_DFA  no_arb_score  n_configs
 Historic -0.075122 -0.081844 -0.068400 0.147962 58         0.256005        0.276549        0.733333        NaN             2         30
Heuristic -0.081971 -0.095541 -0.068400 0.169423 58         1.070327        0.310845        0.800000        NaN             3         30
      CST -0.076577 -0.078637 -0.074827 0.268086 50         1.692360        2.052376        0.533333        NaN             2         30
     CGAN  0.134778  0.083036  0.186519 0.192000 58     51952.555725   247993.487648        0.200000        NaN             1         30
    LobS5 -0.065358 -0.102813 -0.023864 0.253668 58         1.398618        1.477184        0.566667        NaN             2         30
  S5-120M -0.066247 -0.073573 -0.058531 0.132789 56         1.707746        1.752534        0.633333        NaN